# Vecinos aproximados

Compara HNSW, IVF y KNN exacto. Ejecuta todas las celdas. Este notebook contiene su código y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import importlib.util
class IVF:
    """Índice vectorial por clusters, deliberadamente simple para comparar trade-offs."""
    def __init__(self, clusters: int = 32, iterations: int = 8, seed: int = 42):
        self.clusters, self.iterations, self.seed = clusters, iterations, seed

    def fit(self, vectors: np.ndarray, ids: np.ndarray) -> None:
        self.vectors, self.ids = vectors.astype(np.float32), ids
        rng = np.random.default_rng(self.seed)
        centers = self.vectors[rng.choice(len(vectors), min(self.clusters, len(vectors)), replace=False)].copy()
        for _ in range(self.iterations):
            labels = ((self.vectors[:, None] - centers[None, :]) ** 2).sum(axis=2).argmin(axis=1)
            for cluster in range(len(centers)):
                group = self.vectors[labels == cluster]
                if len(group):
                    centers[cluster] = group.mean(axis=0)
        self.centers, self.labels = centers, labels

    def query(self, vector: np.ndarray, k: int, probes: int = 4) -> list[int]:
        chosen = np.argsort(((self.centers - vector) ** 2).sum(axis=1))[:probes]
        positions = np.flatnonzero(np.isin(self.labels, chosen))
        distance = ((self.vectors[positions] - vector) ** 2).sum(axis=1)
        return self.ids[positions[np.argsort(distance)[:k]]].tolist()


def exact_knn(vectors: np.ndarray, ids: np.ndarray, query: np.ndarray, k: int) -> list[int]:
    distance = ((vectors - query) ** 2).sum(axis=1)
    return ids[np.argsort(distance)[:k]].tolist()

def vectors(frame):
    catalog = frame.groupby("product_id").agg(price=("price", "median"), popularity=("user_id", "size")).head(20_000)
    values = catalog.to_numpy(dtype=np.float32)
    values = (values - values.mean(axis=0)) / np.maximum(values.std(axis=0), 1e-8)
    return values, catalog.index.to_numpy(dtype=np.int64)


def run(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    all_vectors, all_ids = vectors(events)
    if len(all_ids) < 10:
        raise RuntimeError("No hay suficientes productos únicos para ANN.")
    for target in sizes(mode):
        limit = min(len(all_ids), max(10, target // 10))
        vec, ids = all_vectors[:limit], all_ids[:limit]
        query, k, reps = vec[0], min(10, len(ids)), repetitions(target, mode)
        truth = exact_knn(vec, ids, query, k)
        exact_time, exact_std, exact_mem = measure(lambda: exact_knn(vec, ids, query, k), reps)
        record(rows, experiment="ANN", algorithm="KNN exacto", n=len(ids), metric="recall_at_k", value=1,
               time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)
        ivf = IVF(clusters=min(32, len(ids)), seed=seed)
        build_time, build_std, build_mem = measure(lambda: ivf.fit(vec, ids), reps)
        ivf.fit(vec, ids)
        candidates = ivf.query(query, k)
        recall = len(set(truth) & set(candidates)) / k
        record(rows, experiment="ANN", algorithm="IVF", n=len(ids), metric="recall_at_k", value=recall,
               time_ms=build_time, std_ms=build_std, memory_mb=build_mem, reps=reps, seed=seed, source=source)
        if importlib.util.find_spec("hnswlib"):
            import hnswlib
            def build_hnsw():
                index = hnswlib.Index(space="l2", dim=vec.shape[1])
                index.init_index(max_elements=len(ids), ef_construction=100, M=16, random_seed=seed)
                index.add_items(vec, ids)
                index.set_ef(50)
                return index
            h_time, h_std, h_mem = measure(build_hnsw, reps)
            index = build_hnsw()
            labels, _ = index.knn_query(query, k=k)
            h_recall = len(set(truth) & set(labels[0])) / k
            record(rows, experiment="ANN", algorithm="HNSW", n=len(ids), metric="recall_at_k", value=h_recall,
                   time_ms=h_time, std_ms=h_std, memory_mb=h_mem, reps=reps, seed=seed, source=source)
        else:
            print("HNSW omitido: instala hnswlib para obtener la comparación completa.")
    path = save(rows, "ann", out)
    result = pd.read_csv(path)
    line_chart(result, "ANN: tiempo", "ann_tiempo.png", out)
    line_chart(result, "ANN: recall@K", "ann_recall.png", out, value="valor", log=False)
    return result

resultado = run(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED)
resultado


## Lectura

KNN exacto fija el recall de referencia. HNSW es la opción del motor cuando mantiene recall alto con una consulta más barata al crecer el catálogo. IVF se deja de lado cuando recupera menos vecinos correctos. Si falta `hnswlib`, el notebook lo indica.
